> **Deprecated.** Stage 2 for inference is CatBoost (`notebooks/modeling/final/train_final_catboost.ipynb`, `infer_catboost.ipynb`). This XGBoost notebook is kept for comparison.


# Inference — final XGBoost pathogenicity classifier

Loads the **stage-2** deployable model (`models/xgb_final_pathogenicity.json` +
`*_meta.json` from `notebooks/train_final_xgboost.ipynb` / `python src/final_model.py`)
and scores **real ClinVar VUS** (uncertain significance) that were cleaned, joined
to UniProt, and position-matched separately from the pathogenic/benign training set.

Honest KPIs live in nested CV (`notebooks/cv_xgboost.ipynb`), not in this notebook.

Build the VUS table first (once):

```bash
python src/clean_clinvar.py
python src/join_clinvar_uniprot.py --vus
python src/position_matching_clinvar_uniprot.py --vus
```


In [1]:
from pathlib import Path
import os
import sys

import pandas as pd
from dotenv import load_dotenv

In [2]:
load_dotenv()
project_root = Path(os.environ["PROJECT_ROOT"])
sys.path.insert(0, str(project_root / "src"))

from final_model import (
    DEFAULT_META_PATH,
    DEFAULT_MODEL_PATH,
    load_final_classifier,
    predict_pathogenicity,
)
from features import (
    POSITION_MATCHED_VUS_IN,
    prepare_inference_frame,
)

INPUT_PARQUET = POSITION_MATCHED_VUS_IN
OUT_PARQUET = project_root / "data/processed/vus_inference_predictions.parquet"

## Load model

In [3]:
model, meta = load_final_classifier(DEFAULT_MODEL_PATH, DEFAULT_META_PATH)
print(f"n_estimators={meta['n_estimators']}, threshold={meta['threshold']:.4f}")
print(f"trained on {meta['n_train']} variants / {meta['n_genes']} genes")
print("features:", meta["feature_columns"])
pd.Series(meta["params"], name="param").to_frame()

n_estimators=79, threshold=0.4894
trained on 11563 variants / 137 genes
features: ['protein_position', 'relative_protein_position', 'distance_to_closest_feature', 'has_protein_position', 'in_domain', 'in_region', 'in_zinc_finger', 'in_active_site', 'in_binding_site', 'in_disulfide', 'in_mod_res', 'in_functional_site', 'in_any_feature', 'closest_feature_type', 'ReferenceAlleleVCF', 'AlternateAlleleVCF']


,param
subsample,0.900000
reg_lambda,1.000000
min_child_weight,10.000000
max_depth,6.000000
learning_rate,0.100000
colsample_bytree,0.800000
scale_pos_weight,0.820372


## Score ClinVar VUS

These variants were **never used as training labels**. Predictions are the model's
pathogenic-vs-benign call at the saved Youden threshold (not a ClinVar reclassification).

In [4]:
raw = pd.read_parquet(INPUT_PARQUET)
frame = prepare_inference_frame(raw, labels={"vus"})
print(f"VUS to score: {len(frame):,} variants / {frame['gene'].nunique():,} genes")
print(frame["gene"].value_counts().head(10))

preds = predict_pathogenicity(frame, model=model, meta=meta)
print("\nPredicted labels:")
print(preds["pred_label"].value_counts())
print(f"\ny_proba: median={preds['y_proba'].median():.3f}, "
      f"mean={preds['y_proba'].mean():.3f}")
preds.sort_values("y_proba", ascending=False).head(15)

VUS to score: 3,506 variants / 123 genes
gene
RUNX1     768
LDLR      300
RYR1      218
PAH       163
HNF1A     151
MYOC      150
HNF4A     100
DICER1     86
GCK        85
ITGA2B     84
Name: count, dtype: int64

Predicted labels:
pred_label
pathogenic    2017
benign        1489
Name: count, dtype: int64

y_proba: median=0.570, mean=0.533


,VariationID,gene,Name,ClinicalSignificance,label,y_proba,threshold,y_pred,pred_label
2619,1691470,ITGB3,NM_000212.3(ITGB3):c.1757G>T (p.Cys586Phe),Uncertain significance,vus,0.966073,0.489401,1,pathogenic
3417,4087789,GP1BA,NM_000173.7(GP1BA):c.790T>C (p.Cys264Arg),Uncertain significance,vus,0.965930,0.489401,1,pathogenic
2631,1693233,PAH,NM_000277.3(PAH):c.667A>T (p.Asn223Tyr),Uncertain significance,vus,0.960520,0.489401,1,pathogenic
3201,3068511,GCK,NM_000162.5(GCK):c.883G>C (p.Gly295Arg),Uncertain significance,vus,0.955934,0.489401,1,pathogenic
220,90844,MSH2,NM_000251.3(MSH2):c.2005G>C (p.Gly669Arg),Uncertain significance,vus,0.953118,0.489401,1,pathogenic
2235,1210180,ITGB3,NM_000212.3(ITGB3):c.1835G>A (p.Cys612Tyr),Uncertain significance,vus,0.952695,0.489401,1,pathogenic
2994,2570626,MYOC,NM_000261.2(MYOC):c.733T>G (p.Cys245Gly),Uncertain significance,vus,0.949550,0.489401,1,pathogenic
2960,2498371,ITGB3,NM_000212.3(ITGB3):c.155G>T (p.Cys52Phe),Uncertain significance,vus,0.949368,0.489401,1,pathogenic
2188,1065373,PAH,NM_000277.3(PAH):c.499A>T (p.Asn167Tyr),Uncertain significance,vus,0.948969,0.489401,1,pathogenic
1132,440548,LDLR,NM_000527.5(LDLR):c.188G>T (p.Cys63Phe),Uncertain significance,vus,0.948388,0.489401,1,pathogenic


In [5]:
OUT_PARQUET.parent.mkdir(parents=True, exist_ok=True)
preds.to_parquet(OUT_PARQUET, index=False)
print("Saved", OUT_PARQUET, preds.shape)

Saved /home/milijanas/projects/clinvar-pathogenicity-classifier/data/processed/vus_inference_predictions.parquet (3506, 9)
